# Feature Engineering : Telco Customer Churn

## Objectif
Transformer les variables brutes en features exploitables par la régression logistique : encodage des variables catégorielles, gestion de la multicolinéarité identifiée, préparation finale du dataset d'entraînement.

## Étapes
1. Chargement du dataset nettoyé
2. Suppression des variables non pertinentes ou redondantes
3. Encodage des variables binaires
4. Encodage one-hot des variables catégorielles multi-classes
5. Encodage de la variable cible
6. Export du dataset final pour la modélisation

In [2]:
import pandas as pd

df = pd.read_csv('../data/processed/telco_cleaned.csv')
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# suppression des variables non pertinentes ou redondantes
df = df.drop(columns=['customerID', 'TotalCharges'])
print(df.shape)

(7043, 19)


In [4]:
# encodage des variables binaires
for col in df.select_dtypes(include='object').columns:
    print(col, ':', df[col].unique())

gender : ['Female' 'Male']
SeniorCitizen : ['No' 'Yes']
Partner : ['Yes' 'No']
Dependents : ['No' 'Yes']
PhoneService : ['No' 'Yes']
MultipleLines : ['No phone service' 'No' 'Yes']
InternetService : ['DSL' 'Fiber optic' 'No']
OnlineSecurity : ['No' 'Yes' 'No internet service']
OnlineBackup : ['Yes' 'No' 'No internet service']
DeviceProtection : ['No' 'Yes' 'No internet service']
TechSupport : ['No' 'Yes' 'No internet service']
StreamingTV : ['No' 'Yes' 'No internet service']
StreamingMovies : ['No' 'Yes' 'No internet service']
Contract : ['Month-to-month' 'One year' 'Two year']
PaperlessBilling : ['Yes' 'No']
PaymentMethod : ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn : ['No' 'Yes']


In [8]:
cols_to_simplify = []

for col in df.select_dtypes(include='object').columns:
    values = set(df[col].unique())
    if values - {'Yes', 'No'}:
        extra_values = values - {'Yes', 'No'}
        if all(str(v).startswith('No ') for v in extra_values):
            cols_to_simplify.append(col)
print(cols_to_simplify)

['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']


In [11]:
for col in cols_to_simplify:
    df[col] = df[col].replace({'No phone service': 'No', 'No internet service': 'No'})
for col in cols_to_simplify:
    print(col, ':', df[col].unique())

MultipleLines : ['No' 'Yes']
OnlineSecurity : ['No' 'Yes']
OnlineBackup : ['Yes' 'No']
DeviceProtection : ['No' 'Yes']
TechSupport : ['No' 'Yes']
StreamingTV : ['No' 'Yes']
StreamingMovies : ['No' 'Yes']


In [12]:
binary_cols = [col for col in df.select_dtypes(include='object').columns
               if df[col].nunique() == 2]
print(binary_cols)

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']


In [13]:
for col in binary_cols:
    print(col, ':', df[col].unique())

gender : ['Female' 'Male']
SeniorCitizen : ['No' 'Yes']
Partner : ['Yes' 'No']
Dependents : ['No' 'Yes']
PhoneService : ['No' 'Yes']
MultipleLines : ['No' 'Yes']
OnlineSecurity : ['No' 'Yes']
OnlineBackup : ['Yes' 'No']
DeviceProtection : ['No' 'Yes']
TechSupport : ['No' 'Yes']
StreamingTV : ['No' 'Yes']
StreamingMovies : ['No' 'Yes']
PaperlessBilling : ['Yes' 'No']
Churn : ['No' 'Yes']


In [14]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
df[binary_cols].head()

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,Churn
0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
1,1,0,0,0,1,0,1,0,1,0,0,0,0,0
2,1,0,0,0,1,0,1,1,0,0,0,0,1,1
3,1,0,0,0,0,0,1,0,1,1,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,1,1


In [15]:
# encodage one-hot des variables catégorielles multi-classes restantes
remaining_categorical = [col for col in df.select_dtypes(include='object').columns]
print(remaining_categorical)

['InternetService', 'Contract', 'PaymentMethod']


In [16]:
df = pd.get_dummies(df, columns=remaining_categorical, drop_first=True)
print(df.shape)

(7043, 23)


In [17]:
# encodage de la variable cible
df['Churn'].unique()

array([0, 1])

In [18]:
df.to_csv('../data/processed/telco_final.csv', index=False)
print("Fichier exporte avec succes :", df.shape)

Fichier exporte avec succes : (7043, 23)
